# ITC Metadata Extraction

This notebook extracts experimental metadata from **all** MicroCal `.itc` (raw thermogram) and `.OPJ` (Origin analysis) files in a given directory.

## MicroCal ITC200 `.itc` Header Format

The header uses three prefix sections:

| Prefix | Content |
|--------|---------|
| `$` | Run config: num injections, set-point temp, reference power, stir speed, feedback mode, injection schedule |
| `#` | Concentration block (7 values): offset, cell-conc-scale, syr-conc-scale, **syringe conc (mM)**, **temperature (°C)**, **cell conc (µM)**, cell volume |
| `%` | Instrument info, calibrated concentrations, baseline coefficients |

**Workflow:**
1. Define parser functions for `.itc` and `.OPJ` formats
2. Discover all files in the target folder
3. Extract metadata from each file
4. Compile into a summary DataFrame

In [1]:
import re
import os

def extract_itc_metadata(filepath):
    """Parses raw parameters from the MicroCal .itc text header.
    
    MicroCal ITC200 header structure:
      $ section  – run configuration (lines 0-28)
        Line 0:  $ITC  (file type)
        Line 1:  $ <num injections>
        Line 3:  $ <set-point temperature, °C>
        Line 4:  $ <reference power, µcal/s>
        Line 5:  $ <initial delay, s>
        Line 7:  $ <stir speed, rpm>  (2 = 750 rpm)
        Lines 10+: $ vol , duration , spacing , filter  (injection schedule)
      # section  – concentration/calibration block (7 lines)
        #0: offset (usually 0)
        #1: cell concentration scale (0.1)
        #2: syringe concentration scale (0.01)
        #3: syringe concentration (mM)
        #4: temperature (°C)
        #5: cell concentration (µM)
        #6: cell volume (µL? or ratio)
      % section  – instrument metadata
        %0: ITC200 firmware version
        %1: syringe concentration (echo)
        %2: cell concentration (echo)
    """
    metadata = {}
    
    if not os.path.exists(filepath):
        return {"Error": f"File {filepath} not found."}
        
    try:
        with open(filepath, 'r', encoding='latin-1') as f:
            lines = f.readlines()
        
        content = ''.join(lines)  # for regex fallbacks
            
        # 1. Identify Instrument
        if 'ITC200' in content:
            metadata['Instrument'] = 'MicroCal ITC200'
            # Extract firmware version
            ver_match = re.search(r'ITC200\s+Ver:\s*([\d.]+)', content)
            if ver_match:
                metadata['Firmware'] = ver_match.group(1)
        
        # 2. Parse $ section (run configuration)
        dollar_lines = []
        for line in lines:
            stripped = line.strip()
            if stripped.startswith('$'):
                dollar_lines.append(stripped[1:].strip())  # remove $ prefix
            elif stripped.startswith('#'):
                break  # $ section ends when # section begins
        
        if len(dollar_lines) >= 4:
            # dollar_lines[1] = num injections, [3] = set-point temp
            try:
                metadata['Set-Point Temp (°C)'] = float(dollar_lines[3])
            except (ValueError, IndexError):
                pass
            try:
                metadata['Reference Power (µcal/s)'] = float(dollar_lines[4])
            except (ValueError, IndexError):
                pass
        
        # 3. Parse # section (concentration & calibration block)
        hash_lines = []
        in_hash = False
        for line in lines:
            stripped = line.strip()
            if stripped.startswith('#'):
                in_hash = True
                hash_lines.append(stripped[1:].strip())  # remove # prefix
            elif in_hash:
                break  # # section ended
        
        if len(hash_lines) >= 7:
            metadata['Syringe Concentration (mM)'] = float(hash_lines[3])
            metadata['Temperature (°C)'] = float(hash_lines[4])
            metadata['Cell Concentration (µM)'] = float(hash_lines[5])
            
        # 4. Extract Injection Parameters from $ lines
        # Injection lines: "vol , duration , spacing , filterPeriod"
        inj_lines = []
        for dl in dollar_lines:
            parts = [p.strip() for p in dl.split(',')]
            if len(parts) == 4:
                try:
                    inj_lines.append(tuple(float(p) for p in parts))
                except ValueError:
                    continue
        
        if inj_lines:
            metadata['Initial Injection Vol (µL)'] = inj_lines[0][0]
            metadata['Initial Spacing (s)'] = inj_lines[0][2]
            
            if len(inj_lines) > 1:
                metadata['Main Injection Vol (µL)'] = inj_lines[1][0]
                metadata['Main Spacing (s)'] = inj_lines[1][2]
                metadata['Total Main Injections'] = len(inj_lines) - 1
                
    except Exception as e:
        metadata['Error'] = f"Failed to parse ITC: {str(e)}"
        
    return metadata

def extract_opj_metadata(filepath):
    """Parses the fitted analysis results directly from the Origin binary .OPJ file."""
    metadata = {}
    
    if not os.path.exists(filepath):
        return {"Error": f"File {filepath} not found."}
        
    try:
        # Read as binary and decode ignoring the binary formatting chunks
        with open(filepath, 'rb') as f:
            raw_data = f.read()
            text_data = raw_data.decode('latin-1', errors='ignore')
            
        # 1. Extract Temperature from OPJ Logs
        temp_match = re.search(r'Temperature:\s*([\d.]+)', text_data)
        if temp_match:
            metadata['Temperature (°C)'] = float(temp_match.group(1))
            
        # 2. Extract Fitting Results Block
        # Origin logs standard outputs as: "Data: ... Model: ... N ... K ... H ... S"
        log_pattern = re.search(
            r'Data:\s*(.*?)\s*'
            r'Model:\s*(\w+)\s*'
            r'Chi\^2/DoF\s*=\s*([\d.E+-]+)\s*'
            r'N\s*([\d.]+)\s*[^\n]*\s*'
            r'K\s*([\d.E+-]+)\s*[^\n]*\s*'
            r'\\g\(D\)H\s*([\d.E+-]+)\s*[^\n]*\s*'
            r'\\g\(D\)S\s*([\d.E+-]+)', 
            text_data, re.IGNORECASE
        )
        
        if log_pattern:
            metadata['Dataset Analyzed'] = log_pattern.group(1).strip()
            metadata['Fitting Model'] = log_pattern.group(2).strip()
            metadata['Chi^2/DoF'] = float(log_pattern.group(3))
            metadata['Stoichiometry (N)'] = float(log_pattern.group(4))
            metadata['Association Constant (Ka)'] = float(log_pattern.group(5))
            
            # Dissociation Constant (Kd) = 1 / Ka
            ka = float(log_pattern.group(5))
            metadata['Dissociation Constant (Kd)'] = f"{(1 / ka):.2E} M" if ka > 0 else "N/A"
            
            metadata['Enthalpy (ΔH) [cal/mol]'] = float(log_pattern.group(6))
            metadata['Entropy (ΔS) [cal/mol/deg]'] = float(log_pattern.group(7))
            
    except Exception as e:
        metadata['Error'] = f"Failed to parse OPJ: {str(e)}"
        
    return metadata

## Batch extraction from all files in the target folder

In [2]:
import pandas as pd
from pathlib import Path

# ── Target directory ──────────────────────────────────────────────
data_dir = Path(
    '/Users/nzlab-la/Library/CloudStorage/OneDrive-TheUniversityofColoradoDenver/'
    'General - Zhao (NZ) Lab/Zhao lab shared folder/Our papers/'
    'Anti-Utag-frankenbody paper/FIg. S2 ITC/20230622_23'
)

# ── Discover files ────────────────────────────────────────────────
itc_files = sorted(data_dir.glob('*.itc'))
opj_files = sorted(data_dir.glob('*.OPJ'))

print(f"Found {len(itc_files)} .itc files and {len(opj_files)} .OPJ files\n")
for f in itc_files:
    print(f"  .itc  → {f.name}")
for f in opj_files:
    print(f"  .OPJ → {f.name}")

Found 7 .itc files and 6 .OPJ files

  .itc  → Q1UTag062123NZ.itc
  .itc  → Q1UTag062123NZ2.itc
  .itc  → Q1UTag062123NZ3.itc
  .itc  → Q1UTag062123NZUTintobuffer.itc
  .itc  → Q1noCUTag062223NZ.itc
  .itc  → Q1noCUTag062223NZ2.itc
  .itc  → Q1noCUTag062223NZ3.itc
  .OPJ → Q1UTag062123NZ.OPJ
  .OPJ → Q1UTag062123NZ2.OPJ
  .OPJ → Q1UTag062123NZ3.OPJ
  .OPJ → Q1noCUTag062223NZ.OPJ
  .OPJ → Q1noCUTag062223NZ2.OPJ
  .OPJ → Q1noCUTag062223NZ3.OPJ


In [3]:
# ── Extract ITC metadata from all .itc files ─────────────────────
itc_records = []
for fpath in itc_files:
    meta = extract_itc_metadata(str(fpath))
    meta['Filename'] = fpath.name
    itc_records.append(meta)

df_itc = pd.DataFrame(itc_records)
# Move Filename to the first column
cols = ['Filename'] + [c for c in df_itc.columns if c != 'Filename']
df_itc = df_itc[cols]

print("=" * 60)
print("RAW ITC METADATA  (.itc files)")
print("=" * 60)
df_itc

RAW ITC METADATA  (.itc files)


,Filename,Instrument,Firmware,Set-Point Temp (°C),Reference Power (µcal/s),Syringe Concentration (mM),Temperature (°C),Cell Concentration (µM),Initial Injection Vol (µL),Initial Spacing (s),Main Injection Vol (µL),Main Spacing (s),Total Main Injections
0,Q1UTag062123NZ.itc,MicroCal ITC200,1.26.1,25.0,600.0,0.2065,25.0,3.8164,0.4,180.0,2.0,180.0,18
1,Q1UTag062123NZ2.itc,MicroCal ITC200,1.26.1,25.0,600.0,0.2065,25.0,3.8164,0.4,180.0,2.0,180.0,18
2,Q1UTag062123NZ3.itc,MicroCal ITC200,1.26.1,25.0,600.0,0.2065,25.0,3.8164,0.4,180.0,2.0,180.0,18
3,Q1UTag062123NZUTintobuffer.itc,MicroCal ITC200,1.26.1,25.0,600.0,0.2065,25.0,3.8164,0.4,180.0,2.0,180.0,18
4,Q1noCUTag062223NZ.itc,MicroCal ITC200,1.26.1,25.0,600.0,0.2065,25.0,3.8164,0.4,180.0,2.0,180.0,18
5,Q1noCUTag062223NZ2.itc,MicroCal ITC200,1.26.1,25.0,600.0,0.2065,25.0,3.8164,0.4,180.0,2.0,180.0,18
6,Q1noCUTag062223NZ3.itc,MicroCal ITC200,1.26.1,25.0,600.0,0.2065,25.0,3.8164,0.4,180.0,2.0,180.0,18


In [4]:
# ── Extract analysis results from all .OPJ files ─────────────────
opj_records = []
for fpath in opj_files:
    meta = extract_opj_metadata(str(fpath))
    meta['Filename'] = fpath.name
    opj_records.append(meta)

df_opj = pd.DataFrame(opj_records)
cols = ['Filename'] + [c for c in df_opj.columns if c != 'Filename']
df_opj = df_opj[cols]

print("=" * 60)
print("ANALYSIS RESULTS  (.OPJ files)")
print("=" * 60)
df_opj

ANALYSIS RESULTS  (.OPJ files)


,Filename,Temperature (°C),Dataset Analyzed,Fitting Model,Chi^2/DoF,Stoichiometry (N),Association Constant (Ka),Dissociation Constant (Kd),Enthalpy (ΔH) [cal/mol],Entropy (ΔS) [cal/mol/deg]
0,Q1UTag062123NZ.OPJ,25.14686,ag062123NZ_NDH,OneSites,612100.0,1.11,34000000.0,2.94E-08 M,-16170.0,-19.8
1,Q1UTag062123NZ2.OPJ,25.04257,g062123NZ2_NDH,OneSites,554600.0,1.10,29900000.0,3.34E-08 M,-16060.0,-19.6
2,Q1UTag062123NZ3.OPJ,25.02383,g062123NZ3_NDH,OneSites,463900.0,1.10,59400000.0,1.68E-08 M,-15710.0,-17.1
3,Q1noCUTag062223NZ.OPJ,25.00298,ag062223NZ_NDH,OneSites,313500.0,1.10,46700000.0,2.14E-08 M,-15630.0,-17.3
4,Q1noCUTag062223NZ2.OPJ,25.00440,g062223NZ2_NDH,OneSites,500400.0,1.21,35200000.0,2.84E-08 M,-15920.0,-18.9
5,Q1noCUTag062223NZ3.OPJ,25.00725,g062223NZ3_NDH,OneSites,2349000.0,1.03,11400000.0,8.77E-08 M,-18690.0,-30.4


In [5]:
# ── Paired summary: merge ITC + OPJ by base filename ─────────────
# Pair files that share the same base name (e.g., Q1UTag062123NZ)
df_itc['BaseName'] = df_itc['Filename'].str.replace('.itc', '', regex=False)
df_opj['BaseName'] = df_opj['Filename'].str.replace('.OPJ', '', regex=False)

df_merged = pd.merge(
    df_itc, df_opj,
    on='BaseName',
    how='outer',
    suffixes=('_itc', '_opj')
)

print("=" * 60)
print("PAIRED SUMMARY  (ITC raw + OPJ analysis)")
print("=" * 60)
print(f"Paired:  {df_merged['BaseName'].nunique()} unique experiments")
print(f"ITC only (no OPJ): {df_merged['Filename_opj'].isna().sum()}")
print(f"OPJ only (no ITC): {df_merged['Filename_itc'].isna().sum()}")
print()
df_merged

PAIRED SUMMARY  (ITC raw + OPJ analysis)
Paired:  7 unique experiments
ITC only (no OPJ): 1
OPJ only (no ITC): 0



,Filename_itc,Instrument,Firmware,Set-Point Temp (°C),Reference Power (µcal/s),Syringe Concentration (mM),Temperature (°C)_itc,Cell Concentration (µM),Initial Injection Vol (µL),Initial Spacing (s),...,Filename_opj,Temperature (°C)_opj,Dataset Analyzed,Fitting Model,Chi^2/DoF,Stoichiometry (N),Association Constant (Ka),Dissociation Constant (Kd),Enthalpy (ΔH) [cal/mol],Entropy (ΔS) [cal/mol/deg]
0,Q1UTag062123NZ.itc,MicroCal ITC200,1.26.1,25.0,600.0,0.2065,25.0,3.8164,0.4,180.0,...,Q1UTag062123NZ.OPJ,25.14686,ag062123NZ_NDH,OneSites,612100.0,1.11,34000000.0,2.94E-08 M,-16170.0,-19.8
1,Q1UTag062123NZ2.itc,MicroCal ITC200,1.26.1,25.0,600.0,0.2065,25.0,3.8164,0.4,180.0,...,Q1UTag062123NZ2.OPJ,25.04257,g062123NZ2_NDH,OneSites,554600.0,1.10,29900000.0,3.34E-08 M,-16060.0,-19.6
2,Q1UTag062123NZ3.itc,MicroCal ITC200,1.26.1,25.0,600.0,0.2065,25.0,3.8164,0.4,180.0,...,Q1UTag062123NZ3.OPJ,25.02383,g062123NZ3_NDH,OneSites,463900.0,1.10,59400000.0,1.68E-08 M,-15710.0,-17.1
3,Q1UTag062123NZUTintobuffer.itc,MicroCal ITC200,1.26.1,25.0,600.0,0.2065,25.0,3.8164,0.4,180.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,Q1noCUTag062223NZ.itc,MicroCal ITC200,1.26.1,25.0,600.0,0.2065,25.0,3.8164,0.4,180.0,...,Q1noCUTag062223NZ.OPJ,25.00298,ag062223NZ_NDH,OneSites,313500.0,1.10,46700000.0,2.14E-08 M,-15630.0,-17.3
5,Q1noCUTag062223NZ2.itc,MicroCal ITC200,1.26.1,25.0,600.0,0.2065,25.0,3.8164,0.4,180.0,...,Q1noCUTag062223NZ2.OPJ,25.00440,g062223NZ2_NDH,OneSites,500400.0,1.21,35200000.0,2.84E-08 M,-15920.0,-18.9
6,Q1noCUTag062223NZ3.itc,MicroCal ITC200,1.26.1,25.0,600.0,0.2065,25.0,3.8164,0.4,180.0,...,Q1noCUTag062223NZ3.OPJ,25.00725,g062223NZ3_NDH,OneSites,2349000.0,1.03,11400000.0,8.77E-08 M,-18690.0,-30.4
